# 00 · pyterraplot overview

**pyterraplot** is an [xarray](https://xarray.dev) accessor + server bridge for the
[terraplot](https://github.com/guidov/terraplot) 3D-globe visualization library. It turns a
2D `DataArray` (lat × lon) into everything terraplot's browser-side `FieldLayer` needs:
JSON payloads, gzip-compressed binary, self-contained HTML globes/maps, animations,
GeoTIFF/COG rasters, and a live HTTP server.

This notebook set gives **full coverage of the public API**. Run them in order, or jump around — each is standalone.

| # | Notebook | Covers |
|---|----------|--------|
| 00 | overview | `import`, accessors, inline `_repr_html_` |
| 01 | serialize / dict / json | `serialize`, `.tp.to_dict`, `.tp.to_json`, dim detection, lon-wrap, NaN |
| 02 | html globe + maps | `.tp.to_html` — 3D globe, 2D projections, contourf, cmap, extent |
| 03 | frames + animation | `.tp.frames`, `frames_compact`, `frames_to_html` |
| 04 | binary packing | `pack_field`, `pack_frames`, the TPLD/TPLF format |
| 05 | dataset accessor | `ds.tp.quiver_html`, `ds.tp.compare_html` |
| 06 | geotiff / cog | `.tp.to_cog` to file and to bytes, round-trip |
| 07 | live server | `.tp.serve` in a background thread |
| 08 | command-line | `python -m pyterraplot` |

> **Prereqs.** Core (numpy, xarray) is required. Optional extras unlock notebooks:
> `pip install 'pyterraplot[serve]'` (07), `'pyterraplot[raster]'` (06), `'pyterraplot[cf]'` (01 bonus).
> HTML rendering needs the terraplot JS bundle — auto-detected from a sibling `../terraplot/dist/terraplot.js`
> checkout, or point `TERRAPLOT_BUNDLE` at it.

## 1. Import registers the `.tp` accessor

Merely importing `pyterraplot` attaches `.tp` to every `xr.DataArray` and `xr.Dataset`.

In [ ]:
import numpy as np
import xarray as xr
import pyterraplot  # registers the .tp accessor on DataArray and Dataset

def make_field(nlat=73, nlon=144, phase=0.0, name="t2m",
               long_name="2m temperature anomaly", units="K", holes=True):
    """A smooth, globe-shaped synthetic field on a regular lat/lon grid."""
    lats = np.linspace(90, -90, nlat)
    lons = np.linspace(-180, 180, nlon)
    LON, LAT = np.meshgrid(lons, lats)
    data = (
        8 * np.cos(np.radians(LAT)) * np.sin(np.radians(2 * LON) + phase)
        + 5 * np.sin(np.radians(3 * LON)) * np.cos(np.radians(2 * LAT))
        + 3 * np.cos(np.radians(5 * LON)) * np.sin(np.radians(LAT))
    ).astype(np.float32)
    if holes:
        rng = np.random.default_rng(0)
        data[rng.random((nlat, nlon)) < 0.02] = np.nan  # NaN "missing" cells
    return xr.DataArray(
        data, dims=["lat", "lon"], coords={"lat": lats, "lon": lons},
        name=name, attrs={"units": units, "long_name": long_name},
    )

da = make_field()
da

## 2. Inline globe via `_repr_html_`

For a **2D** DataArray, displaying it renders an interactive terraplot globe in an iframe (needs the JS bundle). For non-2D arrays it falls back to xarray's normal repr.

Below we display the same `da` explicitly — in JupyterLab the bare `da` at the end of the previous cell already triggers this.

In [ ]:
from IPython.display import HTML, display

try:
    html = da._repr_html_()  # accessor lives on da.tp, but xarray forwards _repr_html_ via the array
except Exception as e:
    html = None
    print("repr failed:", e)

# da._repr_html_ is xarray's; the globe one is on the accessor:
display(HTML(da.tp._repr_html_()))

If you see *“Cannot find terraplot bundle”*, set the bundle path once and re-run:

```python
import os; os.environ['TERRAPLOT_BUNDLE'] = '/path/to/terraplot/dist/terraplot.js'
```

## 3. Whirlwind tour

Everything else in this set, in one cell:

In [ ]:
# dict / json
payload = da.tp.to_dict()
print("payload keys:", list(payload))

# self-contained HTML globe
da.tp.to_html("tour_globe.html", title="tour", cmap="RdYlBu_r")
print("wrote tour_globe.html")

# binary packing
from pyterraplot import pack_field
b64 = pack_field(payload)
print("packed base64 chars:", len(b64))